## Datalab Semester 2, Sprint 3

In [1]:
import os
import sqlite3
import pandas as pd

# 1. Zoek de map op waar dit notebook-bestand staat
map_van_notebook = os.path.dirname(os.path.abspath('__file__'))

# 2. Maak het volledige pad naar de database
db_pad = os.path.join(map_van_notebook, 'database.sqlite')
conn = sqlite3.connect(db_pad)

## Opdracht 1

### 1A


Toon het aantal wedstrijden dat jouw team heeft gespeeld per seizoen.

In [2]:
query = "SELECT name FROM sqlite_master WHERE type='table';"
tabellen = pd.read_sql_query(query, conn)

print(tabellen)
query_zoek_team = "SELECT team_long_name FROM Team WHERE team_long_name LIKE '%Barcelona%';"
barca_naam = pd.read_sql_query(query_zoek_team, conn)

print(barca_naam)

                name
0    sqlite_sequence
1  Player_Attributes
2             Player
3              Match
4             League
5            Country
6               Team
7    Team_Attributes
  team_long_name
0   FC Barcelona


In [3]:
query = """
SELECT 
    season, 
    COUNT(*) AS aantal_wedstrijden
FROM 
    Match
WHERE 
    home_team_api_id = (SELECT team_api_id FROM Team WHERE team_long_name = 'FC Barcelona')
    OR 
    away_team_api_id = (SELECT team_api_id FROM Team WHERE team_long_name = 'FC Barcelona')
GROUP BY 
    season;
"""

df_wedstrijden = pd.read_sql_query(query, conn)
df_wedstrijden

,season,aantal_wedstrijden
0,2008/2009,38
1,2009/2010,38
2,2010/2011,38
3,2011/2012,38
4,2012/2013,38
5,2013/2014,38
6,2014/2015,38
7,2015/2016,38


### 1B

In [4]:
query_year = """
SELECT 
    season,
    COUNT(*) AS aantal_wedstrijden
FROM Match m
JOIN Team t1 ON m.home_team_api_id = t1.team_api_id
JOIN Team t2 ON m.away_team_api_id = t2.team_api_id
WHERE 
    (t1.team_long_name = 'FC Barcelona'
     OR t2.team_long_name = 'FC Barcelona')
    AND strftime('%Y', m.date) = '2010'
GROUP BY season
ORDER BY season;

"""
df_wedstrijden_2010 = pd.read_sql_query(query_year, conn)
df_wedstrijden_2010

,season,aantal_wedstrijden
0,2009/2010,23
1,2010/2011,16


### 1C


In [5]:
def bepaal_match_punten(row):
    """
    Berekent de punten voor de thuis- en uitploeg op basis van de wedstrijdscore.
    
    Args:
        row (pd.Series): Een rij uit de Match dataframe met 'home_team_goal' en 'away_team_goal'.
        
    Returns:
        pd.Series: De behaalde punten voor [home_points, away_points].
    """
    if row['home_team_goal'] > row['away_team_goal']:
        return pd.Series([3, 0], index=['home_points', 'away_points'])
    elif row['home_team_goal'] < row['away_team_goal']:
        return pd.Series([0, 3], index=['home_points', 'away_points'])
    else:
        return pd.Series([1, 1], index=['home_points', 'away_points'])


def bereken_punten_alle_seizoenen(league_id, connection):
    """
    Haalt wedstrijddata op voor een hele competitie en berekent de punten en het doelsaldo per team, per seizoen.
    
    Args:
        league_id (int): De ID van de gekozen competitie.
        connection (sqlite3.Connection): De database verbinding.
        
    Returns:
        pd.DataFrame: Een dataframe met seizoenen, teamnamen, totale punten en doelsaldo.
    """
    # 1. Haal alle wedstrijden op van de competitie
    query = f"SELECT season, home_team_api_id, away_team_api_id, home_team_goal, away_team_goal FROM Match WHERE league_id = {league_id}"
    df_matches = pd.read_sql_query(query, connection)
    
    # 2. De gekopieerde hulpfunctie om de punten per wedstrijd te berekenen
    df_matches[['home_points', 'away_points']] = df_matches.apply(bepaal_match_punten, axis=1)
    
    # 3. De thuis en uitpunten, maar nu PER SEIZOEN en PER TEAM (en inclusief doelpunten!)
    home_stats = df_matches.groupby(['season', 'home_team_api_id'])[['home_points', 'home_team_goal', 'away_team_goal']].sum().reset_index()
    away_stats = df_matches.groupby(['season', 'away_team_api_id'])[['away_points', 'away_team_goal', 'home_team_goal']].sum().reset_index()
    
    home_stats.columns = ['season', 'team_api_id', 'points', 'doelpunten_voor', 'doelpunten_tegen']
    away_stats.columns = ['season', 'team_api_id', 'points', 'doelpunten_voor', 'doelpunten_tegen']
    
    # 4. Plakken thuis en uit onder elkaar.
    alle_stats = pd.concat([home_stats, away_stats])
    # Telt nu punten, doelpunten_voor en doelpunten_tegen bij elkaar op
    ranglijst_seizoenen = alle_stats.groupby(['season', 'team_api_id']).sum().reset_index()
    
    # Doelsaldo berekenen
    ranglijst_seizoenen['doelsaldo'] = ranglijst_seizoenen['doelpunten_voor'] - ranglijst_seizoenen['doelpunten_tegen']
    
    # 5. Clubnamen en de Team tabel
    df_teams_names = pd.read_sql_query("SELECT team_api_id, team_long_name FROM Team", connection)
    ranglijst_seizoenen = ranglijst_seizoenen.merge(df_teams_names, on='team_api_id')
    
    # Eerst op seizoen, dan op punten, en bij gelijke punten op doelsaldo
    eind_ranglijst = ranglijst_seizoenen.sort_values(by=['season', 'points', 'doelsaldo'], ascending=[True, False, False]).reset_index(drop=True)
    
    eind_ranglijst.index = eind_ranglijst.index + 1
    
    return eind_ranglijst[['season', 'team_long_name', 'points', 'doelsaldo']]

In [6]:
# hele League tabel om de ID's te bekijken
query_competities = "SELECT * FROM League"
df_competities = pd.read_sql_query(query_competities, conn)

display(df_competities)

,id,country_id,name
0,1,1,Belgium Jupiler League
1,1729,1729,England Premier League
2,4769,4769,France Ligue 1
3,7809,7809,Germany 1. Bundesliga
4,10257,10257,Italy Serie A
5,13274,13274,Netherlands Eredivisie
6,15722,15722,Poland Ekstraklasa
7,17642,17642,Portugal Liga ZON Sagres
8,19694,19694,Scotland Premier League
9,21518,21518,Spain LIGA BBVA


In [7]:

# ID van de Spain LIGA BBVA
mijn_competitie_id = 21518 

# Zet de rekenmachine aan voor alle seizoenen
df_opdracht_1c = bereken_punten_alle_seizoenen(mijn_competitie_id, conn)

# Filter de dataframe zodat je alleen het gewenste seizoen overhoudt
df_seizoen_2010 = df_opdracht_1c[df_opdracht_1c['season'] == '2010/2011']

# Resultaten printen

display(df_seizoen_2010)


,season,team_long_name,points,doelsaldo
41,2010/2011,FC Barcelona,96,74
42,2010/2011,Real Madrid CF,92,69
43,2010/2011,Valencia CF,71,20
44,2010/2011,Villarreal CF,62,10
45,2010/2011,Atlético Madrid,58,9
46,2010/2011,Athletic Club de Bilbao,58,4
47,2010/2011,Sevilla FC,58,1
48,2010/2011,RCD Espanyol,49,-9
49,2010/2011,CA Osasuna,47,-1
50,2010/2011,Real Sporting de Gijón,47,-7


### 1D




Ons team is geiendigt op de 1ste plaatst binnen de competitie wat in de ranglijst is te zien. 

----------------------------------------------------------------------------------------------


## Opdracht 2

### 2A

In [8]:
query_points = """
SELECT 
    team_id,
    season,
    SUM(points) AS total_points
FROM (
    -- Home wedstrijden
    SELECT 
        home_team_api_id AS team_id,
        season,
        CASE 
            WHEN home_team_goal > away_team_goal THEN 3
            WHEN home_team_goal = away_team_goal THEN 1
            ELSE 0
        END AS points
    FROM Match

    UNION ALL

    -- Away wedstrijden
    SELECT 
        away_team_api_id AS team_id,
        season,
        CASE 
            WHEN away_team_goal > home_team_goal THEN 3
            WHEN away_team_goal = home_team_goal THEN 1
            ELSE 0
        END AS points
    FROM Match
) AS all_matches
GROUP BY team_id, season
"""

df_points = pd.read_sql(query_points, conn)


# 4. TEAM ATTRIBUTES OPHALEN

query_team_attr = """
SELECT 
    team_api_id,
    date,
    buildUpPlaySpeed,
    buildUpPlayPassing,
    chanceCreationPassing,
    chanceCreationCrossing,
    defencePressure,
    defenceAggression
FROM Team_Attributes
"""

df_team_attr = pd.read_sql(query_team_attr, conn)


# 5. DATE → SEASON MAKEN

df_team_attr['date'] = pd.to_datetime(df_team_attr['date'])

df_team_attr['year'] = df_team_attr['date'].dt.year

df_team_attr['season'] = (
    df_team_attr['year'].astype(str) + '/' +
    (df_team_attr['year'] + 1).astype(str)
)


# 6. AGGREGATIE PER SEIZOEN

df_team_attr_grouped = df_team_attr.groupby(
    ['team_api_id', 'season']
)[[
    'buildUpPlaySpeed',
    'buildUpPlayPassing',
    'chanceCreationPassing',
    'chanceCreationCrossing',
    'defencePressure',
    'defenceAggression'
]].mean().reset_index()


# 7. MERGE DATASETS

df_final = pd.merge(
    df_points,
    df_team_attr_grouped,
    left_on=['team_id', 'season'],
    right_on=['team_api_id', 'season'],
    how='inner'
)


print(df_final.head(10))

   team_id     season  total_points  team_api_id  buildUpPlaySpeed  \
0     1601  2010/2011            38         1601              30.0   
1     1601  2011/2012            55         1601              48.0   
2     1601  2012/2013            31         1601              53.0   
3     1601  2013/2014            50         1601              53.0   
4     1601  2014/2015            33         1601              53.0   
5     1601  2015/2016            39         1601              47.0   
6     1773  2012/2013            36         1773              52.0   
7     1957  2010/2011            48         1957              30.0   
8     1957  2011/2012            39         1957              33.0   
9     1957  2012/2013            37         1957              58.0   

   buildUpPlayPassing  chanceCreationPassing  chanceCreationCrossing  \
0                40.0                   50.0                    35.0   
1                51.0                   68.0                    67.0   
2            

In [9]:
query = "PRAGMA table_info(player_Attributes);"
df_info = pd.read_sql_query(query, conn)
df_info

,cid,name,type,notnull,dflt_value,pk
0,0,id,INTEGER,0,None,1
1,1,player_fifa_api_id,INTEGER,0,None,0
2,2,player_api_id,INTEGER,0,None,0
3,3,date,TEXT,0,None,0
4,4,overall_rating,INTEGER,0,None,0
5,5,potential,INTEGER,0,None,0
6,6,preferred_foot,TEXT,0,None,0
7,7,attacking_work_rate,TEXT,0,None,0
8,8,defensive_work_rate,TEXT,0,None,0
9,9,crossing,INTEGER,0,None,0


In [12]:
import pandas as pd

pd.set_option('display.max_columns', None)   # alle kolommen tonen
pd.set_option('display.width', None)         # geen afkapping

query_2 = "SELECT * FROM player_Attributes"
df_player_Attribute = pd.read_sql_query(query_2, conn)

df_player_Attribute.head()

,id,player_fifa_api_id,player_api_id,date,overall_rating,potential,preferred_foot,attacking_work_rate,defensive_work_rate,crossing,finishing,heading_accuracy,short_passing,volleys,dribbling,curve,free_kick_accuracy,long_passing,ball_control,acceleration,sprint_speed,agility,reactions,balance,shot_power,jumping,stamina,strength,long_shots,aggression,interceptions,positioning,vision,penalties,marking,standing_tackle,sliding_tackle,gk_diving,gk_handling,gk_kicking,gk_positioning,gk_reflexes
0,1,218353,505942,2016-02-18 00:00:00,67.0,71.0,right,medium,medium,49.0,44.0,71.0,61.0,44.0,51.0,45.0,39.0,64.0,49.0,60.0,64.0,59.0,47.0,65.0,55.0,58.0,54.0,76.0,35.0,71.0,70.0,45.0,54.0,48.0,65.0,69.0,69.0,6.0,11.0,10.0,8.0,8.0
1,2,218353,505942,2015-11-19 00:00:00,67.0,71.0,right,medium,medium,49.0,44.0,71.0,61.0,44.0,51.0,45.0,39.0,64.0,49.0,60.0,64.0,59.0,47.0,65.0,55.0,58.0,54.0,76.0,35.0,71.0,70.0,45.0,54.0,48.0,65.0,69.0,69.0,6.0,11.0,10.0,8.0,8.0
2,3,218353,505942,2015-09-21 00:00:00,62.0,66.0,right,medium,medium,49.0,44.0,71.0,61.0,44.0,51.0,45.0,39.0,64.0,49.0,60.0,64.0,59.0,47.0,65.0,55.0,58.0,54.0,76.0,35.0,63.0,41.0,45.0,54.0,48.0,65.0,66.0,69.0,6.0,11.0,10.0,8.0,8.0
3,4,218353,505942,2015-03-20 00:00:00,61.0,65.0,right,medium,medium,48.0,43.0,70.0,60.0,43.0,50.0,44.0,38.0,63.0,48.0,60.0,64.0,59.0,46.0,65.0,54.0,58.0,54.0,76.0,34.0,62.0,40.0,44.0,53.0,47.0,62.0,63.0,66.0,5.0,10.0,9.0,7.0,7.0
4,5,218353,505942,2007-02-22 00:00:00,61.0,65.0,right,medium,medium,48.0,43.0,70.0,60.0,43.0,50.0,44.0,38.0,63.0,48.0,60.0,64.0,59.0,46.0,65.0,54.0,58.0,54.0,76.0,34.0,62.0,40.0,44.0,53.0,47.0,62.0,63.0,66.0,5.0,10.0,9.0,7.0,7.0
